In [20]:
%pip install langchain langchain-community langchain-huggingface langchain-pinecone sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
from dotenv import load_dotenv

# Document loading
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector DB

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# LLM
from langchain_groq import ChatGroq

# Prompt + chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [22]:
import os
os.chdir("/Users/tanyabajwa/Desktop/AI-Medical-Chatbot")
print(os.getcwd())

/Users/tanyabajwa/Desktop/AI-Medical-Chatbot


In [3]:
%pip install -U langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

In [5]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [23]:
def load_pdf_files(path):
    loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

documents = load_pdf_files("data")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
texts_chunk = text_splitter.split_documents(documents)

print(f"Chunks created: {len(texts_chunk)}")

Chunks created: 16864


In [24]:
# Embeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Pinecone setup
index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Store vectors
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [25]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

chatModel = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)


In [26]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

system_prompt = """
You are a medical assistant.
Use the context to answer the question.
If unsure, say you don't know.
Keep answer within 3 sentences.

{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | chatModel
    | StrOutputParser()
)

In [27]:
%pip install langchain-huggingface langchain-pinecone sentence-transformers

457.23s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install langchain langchain-community langchain-huggingface langchain-pinecone sentence-transformers pinecone-client

465.63s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)
Using cached pinecone_client-6.0.0-py3-none-any.whl (6.7 kB)
Note: you may need to restart the kernel to use updated packages.


In [29]:
response = rag_chain.invoke("What is Acromegaly and gigantism?")
print(response)


Acromegaly is a condition in which the pituitary gland produces an excess amount of growth hormone, leading to abnormal growth. Gigantism is a condition where an individual grows to an abnormally large size due to excessive growth hormone production, often occurring before the bones have stopped growing, typically in childhood or adolescence.


In [30]:
response = rag_chain.invoke("What is the treatment of Acne?")
print(response)

Acne scars are treated with oral anti-inflammatory medications, and corticosteroid injections, gels, or tapes are used to reduce inflammation and prevent recurrence. Additionally, treatments such as oral isotretinoin may be prescribed for severe cases of acne. These treatments aim to reduce inflammation and prevent further scarring.


In [31]:
response = rag_chain.invoke("What is the treatment of Thyroid?")
print(response)

The treatment of thyroid issues depends on the condition, but for hyperthyroidism, it may involve medication to control symptoms, or in severe cases, a partial thyroidectomy to remove part of the thyroid gland. This surgery is usually considered when medication is ineffective or the condition is life-threatening.
